# 🌿 Détection des Maladies des Plantes — Deep Learning
## ÉTAPE 1/9 : Analyse Exploratoire du Dataset (EDA)

**Projet OFPPT — Deep Learning**
**Dataset :** PlantVillage (Kaggle) — 54 305 images, 38 classes
**Objectif de ce notebook :** comprendre en profondeur le dataset *avant* d'entraîner le moindre modèle.

---

### 📋 Ce que contient ce notebook

| # | Étape | Description |
|---|-------|--------------|
| 1 | Configuration | Installation et import des librairies |
| 2 | Chargement automatique | Téléchargement / détection automatique du dataset |
| 3 | Inventaire | Nombre total d'images et de classes |
| 4 | Répartition par classe | Comptage précis image par image |
| 5 | Visualisation distribution | Graphiques (barres, camembert, déséquilibre) |
| 6 | Exemples visuels | Affichage d'images réelles par classe |
| 7 | Dimensions | Hauteur / largeur / ratio des images |
| 8 | Formats | Extensions de fichiers (.jpg, .png, etc.) |
| 9 | Statistiques pixels | Min / max / moyenne des valeurs de pixels |
| 10 | Images corrompues | Détection automatique des fichiers illisibles |
| 11 | Rapport final | Synthèse complète exportée en fichier texte |
| 12 | Conclusion | Recommandations pour l'étape suivante (CNN from scratch) |

> ⚠️ **Règle du projet :** on ne passe à l'étape 2 (CNN from scratch) que lorsque cette étape 1 est validée à 100%.

> 💡 Chaque cellule de code est précédée d'une cellule Markdown qui explique **pourquoi** on l'exécute et **ce qu'on attend** en sortie.


## 1. ⚙️ Configuration de l'environnement

**Pourquoi cette cellule ?**
On installe/importe toutes les librairies nécessaires à l'analyse :
- `os`, `pathlib` → parcourir les dossiers d'images
- `PIL (Pillow)` → ouvrir, vérifier et inspecter les images
- `numpy`, `pandas` → manipuler les données numériques et tabulaires
- `matplotlib`, `seaborn` → générer des graphiques professionnels
- `tqdm` → afficher des barres de progression (utile avec 54 305 images)
- `kagglehub` → télécharger automatiquement le dataset depuis Kaggle si on est sur Colab

Cette cellule fonctionne **aussi bien sur Google Colab que sur Kaggle Notebook**.


In [ ]:
# Installation des librairies nécessaires (silencieuse si déjà installées)
!pip install -q kagglehub pillow seaborn tqdm

# Import des librairies standards
import os
import sys
import json
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Manipulation de données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec

# Images
from PIL import Image, UnidentifiedImageError

# Barre de progression
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# Configuration esthétique des graphiques (style professionnel)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["font.size"] = 10

# Graine aléatoire pour la reproductibilité (affichage d'exemples)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("✅ Librairies importées avec succès.")
print(f"Python version : {sys.version.split()[0]}")


## 2. 📥 Chargement automatique du dataset

**Pourquoi cette cellule ?**
Le notebook doit fonctionner **sans intervention manuelle**, que tu l'exécutes sur :
- **Kaggle Notebook** → le dataset est déjà monté dans `/kaggle/input/plantvillage-dataset/color`
- **Google Colab** → il faut le télécharger via `kagglehub` (l'API officielle de Kaggle)

La cellule détecte automatiquement l'environnement d'exécution et agit en conséquence :
1. Si le chemin Kaggle existe déjà → on l'utilise directement (rapide, pas de téléchargement).
2. Sinon (Colab ou local) → on télécharge le dataset PlantVillage via `kagglehub.dataset_download(...)`.

> 🔑 **Sur Colab**, il te faudra un compte Kaggle + un token API (`kaggle.json`).
> `kagglehub` ouvrira automatiquement une fenêtre d'authentification si nécessaire — aucune manipulation de fichier requise.


In [ ]:
# Chemin attendu sur Kaggle Notebook
KAGGLE_PATH = "/kaggle/input/plantvillage-dataset/color"

if os.path.exists(KAGGLE_PATH):
    # Cas 1 : on est déjà sur Kaggle Notebook, le dataset est monté
    DATA_DIR = KAGGLE_PATH
    print(f"✅ Environnement détecté : Kaggle Notebook")
    print(f"📁 Dataset déjà disponible : {DATA_DIR}")

else:
    # Cas 2 : on est sur Google Colab (ou en local) -> téléchargement via kagglehub
    print("✅ Environnement détecté : Google Colab / Local")
    print("⬇️  Téléchargement du dataset PlantVillage depuis Kaggle...")

    import kagglehub
    # Téléchargement (mis en cache automatiquement si déjà téléchargé)
    dataset_root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

    # Le dataset contient plusieurs sous-dossiers (color / grayscale / segmented)
    # -> on utilise la version "color" (images RGB), conformément au cahier des charges
    DATA_DIR = os.path.join(dataset_root, "color")

    print(f"📁 Dataset téléchargé et disponible ici : {DATA_DIR}")

# Vérification finale
assert os.path.exists(DATA_DIR), f"❌ Le chemin {DATA_DIR} n'existe pas. Vérifie le téléchargement."
print("\n✅ Dataset prêt à être analysé.")


## 3. 🔢 Inventaire global : nombre d'images et de classes

**Pourquoi cette cellule ?**
Avant toute analyse fine, on vérifie que le dataset correspond bien à ce qui est annoncé dans le cahier des charges :
- **54 305 images** attendues
- **38 classes** attendues (1 dossier = 1 classe, ex : `Apple___Apple_scab`, `Tomato___healthy`...)

On parcourt récursivement chaque dossier de classe et on compte les fichiers images valides (`.jpg`, `.jpeg`, `.png`, `.JPG`, `.JPEG`, `.PNG`).


In [ ]:
# Extensions d'images valides
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

# Liste des classes = liste des sous-dossiers du dataset
class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(f"📦 Nombre total de classes détectées : {len(class_names)}")

# Construction d'un dictionnaire {classe: [chemins des images]}
class_to_filepaths = {}

for class_name in tqdm(class_names, desc="Inventaire des classes"):
    class_dir = os.path.join(DATA_DIR, class_name)
    filepaths = [
        os.path.join(class_dir, f)
        for f in os.listdir(class_dir)
        if f.endswith(VALID_EXTENSIONS)
    ]
    class_to_filepaths[class_name] = filepaths

# Nombre total d'images
total_images = sum(len(v) for v in class_to_filepaths.values())

print(f"\n🖼️  Nombre total d'images trouvées : {total_images}")
print(f"📁 Nombre total de classes : {len(class_names)}")

# Comparaison avec le cahier des charges
expected_images, expected_classes = 54305, 38
print("\n--- Comparaison avec le cahier des charges ---")
print(f"Images attendues : {expected_images} | trouvées : {total_images} | "
      f"écart : {total_images - expected_images}")
print(f"Classes attendues : {expected_classes} | trouvées : {len(class_names)} | "
      f"écart : {len(class_names) - expected_classes}")


## 4. 📊 Répartition détaillée par classe

**Pourquoi cette cellule ?**
On construit un tableau (`DataFrame` pandas) qui liste, pour **chaque classe** :
- le nombre d'images,
- son pourcentage par rapport au total.

Cela permet de repérer immédiatement :
- les classes **sur-représentées** (risque de biais du modèle vers ces classes),
- les classes **sous-représentées** (risque que le modèle ne les apprenne pas bien — déséquilibre de classes).


In [ ]:
# Construction du DataFrame de comptage
counts_data = [
    {"classe": cls, "nb_images": len(paths)}
    for cls, paths in class_to_filepaths.items()
]
df_counts = pd.DataFrame(counts_data).sort_values("nb_images", ascending=False).reset_index(drop=True)
df_counts["pourcentage"] = (df_counts["nb_images"] / df_counts["nb_images"].sum() * 100).round(2)

# Extraction de l'espèce et du statut (sain / malade) à partir du nom de la classe
# Exemple : "Tomato___Early_blight" -> espèce = "Tomato", statut = "Early_blight"
df_counts["espece"] = df_counts["classe"].apply(lambda x: x.split("___")[0])
df_counts["statut"] = df_counts["classe"].apply(
    lambda x: "healthy" if "healthy" in x.lower() else "malade"
)

print("📊 Statistiques de répartition par classe :")
print(df_counts["nb_images"].describe().round(2))

print("\n🔝 Top 5 classes les plus représentées :")
display(df_counts.head(5)[["classe", "nb_images", "pourcentage"]])

print("\n🔻 Top 5 classes les moins représentées :")
display(df_counts.tail(5)[["classe", "nb_images", "pourcentage"]])

# Ratio déséquilibre = classe la plus grande / classe la plus petite
imbalance_ratio = df_counts["nb_images"].max() / df_counts["nb_images"].min()
print(f"\n⚖️  Ratio de déséquilibre (max/min) : {imbalance_ratio:.2f}x")


## 5. 📈 Visualisation de la distribution des classes

**Pourquoi cette cellule ?**
Un graphique est beaucoup plus parlant qu'un tableau de 38 lignes. On génère ici **trois visualisations professionnelles** :
1. Un **diagramme en barres horizontales** (toutes les classes triées) → vue détaillée.
2. Un **diagramme en barres par espèce** (sain vs malade agrégé) → vue synthétique.
3. Une **boîte à moustaches (boxplot)** du nombre d'images par classe → vue statistique du déséquilibre.


In [ ]:
fig = plt.figure(figsize=(16, 18))
gs = gridspec.GridSpec(3, 1, height_ratios=[2.2, 1, 0.6], hspace=0.5)

# --- Graphique 1 : barres horizontales pour les 38 classes ---
ax1 = fig.add_subplot(gs[0])
colors = sns.color_palette("viridis", len(df_counts))
sns.barplot(
    data=df_counts.sort_values("nb_images"),
    y="classe", x="nb_images", hue="classe",
    palette=colors, legend=False, ax=ax1
)
ax1.set_title("Distribution du nombre d'images par classe (38 classes)", fontsize=15)
ax1.set_xlabel("Nombre d'images")
ax1.set_ylabel("Classe")
ax1.axvline(df_counts["nb_images"].mean(), color="red", linestyle="--", linewidth=1.5,
            label=f"Moyenne ({df_counts['nb_images'].mean():.0f})")
ax1.legend(loc="lower right")

# --- Graphique 2 : agrégation par espèce ---
ax2 = fig.add_subplot(gs[1])
species_counts = df_counts.groupby("espece")["nb_images"].sum().sort_values(ascending=False)
sns.barplot(x=species_counts.values, y=species_counts.index, hue=species_counts.index,
            palette="mako", legend=False, ax=ax2)
ax2.set_title("Nombre total d'images par espèce de plante", fontsize=15)
ax2.set_xlabel("Nombre d'images")
ax2.set_ylabel("Espèce")

# --- Graphique 3 : boxplot du déséquilibre ---
ax3 = fig.add_subplot(gs[2])
sns.boxplot(x=df_counts["nb_images"], color="skyblue", ax=ax3)
sns.stripplot(x=df_counts["nb_images"], color="darkred", alpha=0.6, size=5, ax=ax3)
ax3.set_title("Dispersion du nombre d'images par classe (déséquilibre)", fontsize=15)
ax3.set_xlabel("Nombre d'images")

plt.tight_layout()
plt.savefig("distribution_classes.png", dpi=150, bbox_inches="tight")
plt.show()

print("💾 Graphique sauvegardé : distribution_classes.png")


In [ ]:
# --- Graphique complémentaire : répartition sain vs malade ---
plt.figure(figsize=(7, 7))
status_counts = df_counts.groupby("statut")["nb_images"].sum()
colors_pie = ["#4CAF50", "#E53935"]  # vert = healthy, rouge = malade
plt.pie(
    status_counts.values,
    labels=[f"{idx} ({val:,} images)" for idx, val in status_counts.items()],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors_pie,
    explode=[0.03] * len(status_counts),
    textprops={"fontsize": 11},
)
plt.title("Proportion globale : feuilles saines vs malades", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("repartition_sain_malade.png", dpi=150, bbox_inches="tight")
plt.show()

print("💾 Graphique sauvegardé : repartition_sain_malade.png")


## 6. 🖼️ Affichage d'exemples d'images

**Pourquoi cette cellule ?**
Voir des images réelles permet de :
- vérifier visuellement la qualité et la cohérence du dataset,
- repérer d'éventuelles anomalies (images floues, mal recadrées, mauvaise classe...),
- comprendre à quoi ressemblent les symptômes des maladies à détecter.

On affiche **une image aléatoire pour chacune des 38 classes**, organisées en grille.


In [ ]:
n_cols = 6
n_rows = int(np.ceil(len(class_names) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3.2 * n_rows))
axes = axes.flatten()

for i, class_name in enumerate(class_names):
    img_path = random.choice(class_to_filepaths[class_name])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(class_name, fontsize=8)
    axes[i].axis("off")

# On masque les sous-graphiques inutilisés
for j in range(len(class_names), len(axes)):
    axes[j].axis("off")

plt.suptitle("Exemple d'image pour chacune des 38 classes", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("exemples_par_classe.png", dpi=150, bbox_inches="tight")
plt.show()

print("💾 Grille d'exemples sauvegardée : exemples_par_classe.png")


In [ ]:
# Zoom : plusieurs exemples pour UNE classe précise (ex: la première classe disponible)
zoom_class = class_names[0]
sample_paths = random.sample(class_to_filepaths[zoom_class], min(8, len(class_to_filepaths[zoom_class])))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.flatten(), sample_paths):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f"{img.size[0]}x{img.size[1]}px", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Zoom sur la classe : {zoom_class}", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. 📐 Vérification des dimensions des images

**Pourquoi cette cellule ?**
Les modèles CNN attendent une taille d'image **fixe** en entrée. Il faut donc savoir :
- si toutes les images ont la même taille,
- ou s'il existe une grande variabilité (ce qui justifiera un `resize` systématique au moment du prétraitement).

⚠️ Pour des raisons de performance (54 305 images), on analyse un **échantillon représentatif** (toutes les classes, N images par classe) plutôt que la totalité — c'est une pratique standard en EDA. Tu peux augmenter `SAMPLE_PER_CLASS` si tu veux une analyse exhaustive (plus lent).


In [ ]:
SAMPLE_PER_CLASS = 60  # nombre d'images analysées par classe (échantillon représentatif)

dimensions = []  # (width, height)
formats_found = []
modes_found = []

sampled_filepaths = []
for class_name, paths in class_to_filepaths.items():
    sampled_filepaths.extend(random.sample(paths, min(SAMPLE_PER_CLASS, len(paths))))

print(f"🔍 Analyse d'un échantillon de {len(sampled_filepaths)} images "
      f"({SAMPLE_PER_CLASS} max par classe sur {len(class_names)} classes)...")

for path in tqdm(sampled_filepaths, desc="Analyse des dimensions/formats"):
    try:
        with Image.open(path) as img:
            dimensions.append(img.size)        # (width, height)
            formats_found.append(img.format)   # JPEG, PNG...
            modes_found.append(img.mode)       # RGB, RGBA, L...
    except Exception as e:
        print(f"⚠️ Erreur de lecture sur {path} : {e}")

widths = [d[0] for d in dimensions]
heights = [d[1] for d in dimensions]
unique_dims = Counter(dimensions)

print(f"\n📐 Dimensions uniques trouvées : {len(unique_dims)}")
print("Top 5 dimensions les plus fréquentes :")
for dim, count in unique_dims.most_common(5):
    print(f"   {dim[0]}x{dim[1]} px  →  {count} images ({count/len(dimensions)*100:.1f}%)")

print(f"\nLargeur  -> min: {min(widths)}px | max: {max(widths)}px | moyenne: {np.mean(widths):.1f}px")
print(f"Hauteur  -> min: {min(heights)}px | max: {max(heights)}px | moyenne: {np.mean(heights):.1f}px")

if len(unique_dims) == 1:
    print("\n✅ Toutes les images de l'échantillon ont EXACTEMENT la même taille — pas de resize obligatoire (mais recommandé pour homogénéité future).")
else:
    print("\n⚠️ Les images ont des tailles VARIABLES — un `resize` systématique sera nécessaire avant l'entraînement du CNN.")


In [ ]:
# Visualisation de la distribution des dimensions (largeur x hauteur)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(widths, bins=20, kde=True, color="seagreen", ax=axes[0])
axes[0].set_title("Distribution des largeurs (px)")
axes[0].set_xlabel("Largeur (px)")

sns.histplot(heights, bins=20, kde=True, color="steelblue", ax=axes[1])
axes[1].set_title("Distribution des hauteurs (px)")
axes[1].set_xlabel("Hauteur (px)")

plt.tight_layout()
plt.savefig("distribution_dimensions.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. 🗂️ Vérification des formats d'images

**Pourquoi cette cellule ?**
On vérifie :
- le **format réel du fichier** détecté par Pillow (JPEG, PNG...) — qui doit correspondre à l'extension du fichier,
- le **mode couleur** (`RGB`, `RGBA`, `L` pour niveaux de gris...) — un CNN couleur attend du `RGB` à 3 canaux ; une image `RGBA` (4 canaux) ou `L` (1 canal, gris) devra être convertie.


In [ ]:
format_counts = Counter(formats_found)
mode_counts = Counter(modes_found)
extension_counts = Counter(Path(p).suffix.lower() for p in sampled_filepaths)

print("📦 Formats d'image détectés (Pillow) :")
for fmt, count in format_counts.most_common():
    print(f"   {fmt:<10} → {count} images ({count/len(formats_found)*100:.1f}%)")

print("\n🎨 Modes couleur détectés :")
for mode, count in mode_counts.most_common():
    print(f"   {mode:<10} → {count} images ({count/len(modes_found)*100:.1f}%)")

print("\n📄 Extensions de fichiers détectées :")
for ext, count in extension_counts.most_common():
    print(f"   {ext:<10} → {count} images ({count/len(sampled_filepaths)*100:.1f}%)")

if len(mode_counts) == 1 and "RGB" in mode_counts:
    print("\n✅ Toutes les images sont en RGB (3 canaux) — cohérent avec une classification d'images couleur.")
else:
    print("\n⚠️ Modes couleur hétérogènes détectés — une conversion systématique en RGB sera nécessaire (`img.convert('RGB')`).")


## 9. 🎚️ Vérification des valeurs min/max des pixels

**Pourquoi cette cellule ?**
Avant d'entraîner un CNN, il faut savoir comment les pixels sont encodés :
- normalement, les valeurs de pixels sont des entiers compris entre **0 et 255** (image 8 bits),
- si certaines images ont des valeurs hors de cette plage, ou des canaux constants (ex : tout à 0), cela peut indiquer des images corrompues ou anormales,
- cette information confirme aussi la stratégie de normalisation à appliquer plus tard (ex : diviser par 255 pour ramener les valeurs entre 0 et 1).


In [ ]:
pixel_mins, pixel_maxs, pixel_means = [], [], []

# On réutilise un sous-échantillon (un peu plus petit, car charger les pixels coûte plus cher)
PIXEL_SAMPLE_PER_CLASS = 15
pixel_sample_paths = []
for class_name, paths in class_to_filepaths.items():
    pixel_sample_paths.extend(random.sample(paths, min(PIXEL_SAMPLE_PER_CLASS, len(paths))))

for path in tqdm(pixel_sample_paths, desc="Analyse des valeurs de pixels"):
    try:
        with Image.open(path) as img:
            arr = np.array(img.convert("RGB"))
            pixel_mins.append(arr.min())
            pixel_maxs.append(arr.max())
            pixel_means.append(arr.mean())
    except Exception as e:
        print(f"⚠️ Erreur sur {path} : {e}")

print(f"🔢 Analyse sur {len(pixel_sample_paths)} images :")
print(f"Valeur de pixel minimale observée  : {min(pixel_mins)}")
print(f"Valeur de pixel maximale observée  : {max(pixel_maxs)}")
print(f"Moyenne globale des pixels         : {np.mean(pixel_means):.2f}")
print(f"Plage théorique attendue           : [0, 255] (image 8 bits)")

if min(pixel_mins) >= 0 and max(pixel_maxs) <= 255:
    print("\n✅ Toutes les valeurs de pixels sont dans la plage attendue [0, 255].")
else:
    print("\n⚠️ Valeurs de pixels hors plage détectées — à investiguer.")

print("\n👉 Normalisation recommandée pour l'entraînement : pixel / 255.0  → plage [0, 1]")


In [ ]:
# Histogramme de la distribution des intensités de pixels (échantillon)
plt.figure(figsize=(10, 5))
sns.histplot(pixel_means, bins=30, kde=True, color="purple")
plt.title("Distribution de l'intensité moyenne des pixels par image (échantillon)")
plt.xlabel("Intensité moyenne des pixels (0-255)")
plt.ylabel("Nombre d'images")
plt.tight_layout()
plt.savefig("distribution_pixels.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. 🚨 Détection des images corrompues

**Pourquoi cette cellule ?**
Sur un dataset de plus de 54 000 images, il est fréquent de trouver :
- des fichiers tronqués (téléchargement interrompu),
- des fichiers à l'extension trompeuse (ex : un `.png` qui n'est pas une vraie image PNG),
- des fichiers de taille nulle (0 octet).

On utilise `Image.verify()` de Pillow, qui lit l'en-tête du fichier sans charger toute l'image en mémoire (rapide), puis on tente une ouverture complète pour confirmer.

> ⏱️ Cette vérification passe en revue **TOUTES les images** du dataset (54 305) — cela peut prendre quelques minutes, mais c'est indispensable avant tout entraînement.


In [ ]:
corrupted_files = []
all_filepaths = [p for paths in class_to_filepaths.values() for p in paths]

for path in tqdm(all_filepaths, desc="Vérification de l'intégrité des images"):
    try:
        # Vérification de taille nulle
        if os.path.getsize(path) == 0:
            corrupted_files.append((path, "Fichier vide (0 octet)"))
            continue

        # Vérification d'intégrité Pillow
        with Image.open(path) as img:
            img.verify()

        # Re-ouverture complète pour s'assurer que le décodage fonctionne réellement
        with Image.open(path) as img:
            img.load()

    except (UnidentifiedImageError, OSError, SyntaxError) as e:
        corrupted_files.append((path, str(e)))

print(f"\n🔍 {len(all_filepaths)} images vérifiées.")
print(f"🚨 Images corrompues détectées : {len(corrupted_files)}")

if corrupted_files:
    print("\nListe des fichiers problématiques (max 20 affichés) :")
    for path, error in corrupted_files[:20]:
        print(f"   ❌ {path}  →  {error}")
else:
    print("✅ Aucune image corrompue détectée — le dataset est intègre à 100%.")


## 11. 📝 Rapport complet d'analyse des données

**Pourquoi cette cellule ?**
On rassemble **tous les résultats précédents** dans un rapport unique, structuré et exportable. Ce rapport servira de référence pour :
- justifier les choix de prétraitement à l'étape 2 (resize, normalisation, conversion RGB...),
- documenter le projet dans le rapport PDF final (étape 8 du cahier des charges).

Le rapport est affiché dans le notebook **et** sauvegardé dans un fichier `rapport_exploration_dataset.txt`.


In [ ]:
report_lines = []
def add(line=""):
    report_lines.append(line)

add("=" * 70)
add("RAPPORT D'ANALYSE EXPLORATOIRE — DATASET PLANTVILLAGE")
add("=" * 70)
add(f"Chemin du dataset : {DATA_DIR}")
add("")
add("1. INVENTAIRE GLOBAL")
add("-" * 70)
add(f"Nombre total d'images       : {total_images}")
add(f"Nombre total de classes     : {len(class_names)}")
add(f"Images attendues (cahier)   : {expected_images}")
add(f"Classes attendues (cahier)  : {expected_classes}")
add("")
add("2. REPARTITION PAR CLASSE")
add("-" * 70)
add(f"Classe la plus représentée  : {df_counts.iloc[0]['classe']} ({df_counts.iloc[0]['nb_images']} images)")
add(f"Classe la moins représentée : {df_counts.iloc[-1]['classe']} ({df_counts.iloc[-1]['nb_images']} images)")
add(f"Moyenne d'images / classe   : {df_counts['nb_images'].mean():.1f}")
add(f"Ecart-type                  : {df_counts['nb_images'].std():.1f}")
add(f"Ratio de déséquilibre       : {imbalance_ratio:.2f}x")
add(f"Nombre d'espèces de plantes : {df_counts['espece'].nunique()}")
add("")
add("3. DIMENSIONS DES IMAGES (échantillon)")
add("-" * 70)
add(f"Dimensions uniques trouvées : {len(unique_dims)}")
add(f"Dimension la plus fréquente : {unique_dims.most_common(1)[0][0]} px")
add(f"Largeur min/max/moyenne     : {min(widths)} / {max(widths)} / {np.mean(widths):.1f} px")
add(f"Hauteur min/max/moyenne     : {min(heights)} / {max(heights)} / {np.mean(heights):.1f} px")
add("")
add("4. FORMATS ET MODES COULEUR")
add("-" * 70)
add(f"Formats détectés  : {dict(format_counts)}")
add(f"Modes couleur     : {dict(mode_counts)}")
add(f"Extensions        : {dict(extension_counts)}")
add("")
add("5. VALEURS DE PIXELS")
add("-" * 70)
add(f"Min global : {min(pixel_mins)}  |  Max global : {max(pixel_maxs)}")
add(f"Intensité moyenne : {np.mean(pixel_means):.2f}")
add(f"Normalisation recommandée : division par 255.0")
add("")
add("6. INTEGRITE DU DATASET")
add("-" * 70)
add(f"Images vérifiées    : {len(all_filepaths)}")
add(f"Images corrompues   : {len(corrupted_files)}")
if corrupted_files:
    add("Liste des fichiers corrompus :")
    for path, error in corrupted_files:
        add(f"   - {path} ({error})")
add("")
add("7. RECOMMANDATIONS POUR L'ÉTAPE SUIVANTE (CNN From Scratch)")
add("-" * 70)
add("- Resize systématique de toutes les images vers une taille fixe (ex: 224x224 ou 128x128)")
add("- Conversion systématique en RGB (img.convert('RGB'))")
add("- Normalisation des pixels : division par 255.0 (plage [0,1])")
add("- Gestion du déséquilibre de classes : class_weight, data augmentation ciblée, ou oversampling")
add("- Split stratifié train/val/test pour préserver la proportion des classes")
add("- Exclusion des images corrompues identifiées ci-dessus (si applicable)")
add("=" * 70)

report_text = "\n".join(report_lines)
print(report_text)

with open("rapport_exploration_dataset.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n💾 Rapport sauvegardé : rapport_exploration_dataset.txt")

# Sauvegarde additionnelle du tableau de comptage par classe (réutilisable étape 2)
df_counts.to_csv("repartition_classes.csv", index=False)
print("💾 Tableau des classes sauvegardé : repartition_classes.csv")


## 12. ✅ Conclusion de l'étape 1

**Ce que nous avons accompli :**
- ✅ Chargement automatique et vérifié du dataset PlantVillage (Kaggle ou Colab)
- ✅ Inventaire complet : nombre d'images, nombre de classes
- ✅ Analyse fine de la répartition par classe (équilibrée ou non)
- ✅ Visualisations professionnelles de la distribution
- ✅ Exemples visuels pour les 38 classes
- ✅ Vérification des dimensions, formats et modes couleur
- ✅ Vérification des valeurs de pixels (plage [0, 255])
- ✅ Détection exhaustive des images corrompues
- ✅ Rapport complet exporté (`.txt` + `.csv`)

**📌 Fichiers générés par ce notebook :**
- `distribution_classes.png`
- `repartition_sain_malade.png`
- `exemples_par_classe.png`
- `distribution_dimensions.png`
- `distribution_pixels.png`
- `rapport_exploration_dataset.txt`
- `repartition_classes.csv`

---

### ➡️ Prochaine étape (Étape 2/9)
**Développement d'un modèle CNN From Scratch.**
Nous ne passerons à cette étape que lorsque tu auras validé les résultats de cette analyse exploratoire.

N'hésite pas à me dire :
- si le dataset chez toi correspond bien aux chiffres attendus (54 305 images / 38 classes),
- si tu repères un déséquilibre que tu veux traiter différemment,
- et je préparerai alors le notebook de l'étape 2 en tenant compte de ces résultats.
